# User and expert evaluation
Use only real responses collected under the study protocol. Technical tests do not establish usability, clinical suitability, or increased help-seeking.
Run after notebook 02 and repeat the functional checks on the hosted app.

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Final Research - C D Jayasinghe')
for folder in ['data', 'models', 'outputs', 'report_figures', 'deployment', 'notebooks']:
    (ROOT / folder).mkdir(parents=True, exist_ok=True)

## 1. Hosted browser acceptance
Check desktop and a 390-pixel-wide mobile viewport: no preselected answers; eligibility required; missing answers rejected; valid completion shows three probabilities summing to 1; explanations displayed; support information available for every class; refresh clears answers; leave-page link works.
Save results in `outputs/hosted_acceptance.csv` with columns `case`, `passed` (0/1), `notes`. Record defects without entering personal survey responses.

In [ ]:
import pandas as pd
path=ROOT/'outputs/hosted_acceptance.csv'
if path.exists():
    tests=pd.read_csv(path)
    required={'case','passed','notes'}
    if not required.issubset(tests.columns) or tests.empty or not tests['passed'].isin([0,1]).all():
        raise ValueError('Acceptance records need case, passed (0/1), and notes.')
    display(tests)
    print('Hosted acceptance pass rate:',round(tests['passed'].mean()*100,1),'%')
else:
    print('Pending hosted browser evaluation.')

## 2. System Usability Scale (SUS)
After representative users complete an assessment and interpret its result, collect the standard ten SUS items, rated 1 (strongly disagree) to 5 (strongly agree):
1. I think that I would like to use this system frequently.
2. I found the system unnecessarily complex.
3. I thought the system was easy to use.
4. I think that I would need the support of a technical person to be able to use this system.
5. I found the various functions in this system were well integrated.
6. I thought there was too much inconsistency in this system.
7. I would imagine that most people would learn to use this system very quickly.
8. I found the system very cumbersome to use.
9. I felt very confident using the system.
10. I needed to learn a lot of things before I could get going with this system.

Save anonymous responses in `outputs/sus_responses.csv`, columns `SUS1`–`SUS10`. SUS is a score from 0–100, not a percentage of usability. Do not impute incomplete responses.

In [ ]:
import numpy as np
path=ROOT/'outputs/sus_responses.csv'
if path.exists():
    responses=pd.read_csv(path)
    columns=[f'SUS{i}' for i in range(1,11)]
    if not set(columns).issubset(responses.columns):
        raise ValueError('Expected SUS1-SUS10.')
    values=responses[columns].apply(pd.to_numeric,errors='raise')
    invalid=values.notna() & (~values.isin([1,2,3,4,5]))
    if invalid.any().any(): raise ValueError('SUS responses must be integers 1-5.')
    complete=values.dropna()
    if complete.empty: raise ValueError('No complete SUS responses.')
    scores=((complete.iloc[:,::2]-1).sum(axis=1)+(5-complete.iloc[:,1::2]).sum(axis=1))*2.5
    summary=pd.Series({'submitted_n':len(values),'complete_n':len(scores),'excluded_incomplete_n':len(values)-len(scores),
                       'mean_sus':scores.mean(),'sd_sus':scores.std(),'median_sus':scores.median(),
                       'minimum_sus':scores.min(),'maximum_sus':scores.max()})
    summary.to_csv(ROOT/'outputs/sus_summary.csv',header=['Value']); display(summary)
else:
    print('Pending real user SUS responses.')

## 3. Expert review
Ask qualified domain experts to review questionnaire wording, explanation clarity, interpretation of intention, support guidance, cultural suitability, and privacy. Record technical reviewers separately.
Save `outputs/expert_reviews.csv` with anonymous `reviewer_id`, `role` (`domain`/`technical`), `criterion`, `rating` (1–5), `comment`, and `required_change`. Summaries describe reviewer judgements; they are not clinical validation.

In [ ]:
path=ROOT/'outputs/expert_reviews.csv'
if path.exists():
    reviews=pd.read_csv(path)
    required={'reviewer_id','role','criterion','rating','comment','required_change'}
    if not required.issubset(reviews.columns) or reviews.empty:
        raise ValueError('Missing expert review columns or no responses.')
    if not reviews['role'].isin(['domain','technical']).all() or not reviews['rating'].isin([1,2,3,4,5]).all():
        raise ValueError('Check reviewer roles and ratings.')
    if reviews.duplicated(['reviewer_id','role','criterion']).any():
        raise ValueError('Duplicate reviewer/criterion records.')
    summary=reviews.groupby(['role','criterion'])['rating'].agg(['count','mean','median','min','max'])
    summary.to_csv(ROOT/'outputs/expert_summary.csv'); display(summary)
    display(reviews.loc[reviews['required_change'].fillna('').str.strip().ne(''),['role','criterion','required_change']])
else:
    print('Pending real expert reviews.')